
# Segmentation

* fasc




In [1]:
from PIL import Image
import os
import hashlib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import time
from datetime import datetime, timezone, timedelta

try:
    from google.colab import drive
    from google.colab import runtime
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:

    drive.mount("/content/drive")

    !mkdir -p /content/my_dataset
    !unzip -q -o "/content/drive/MyDrive/colab/UMUD/umud-challenge-muscle-architecture-in-ultrasound-data.zip" -d /content/my_dataset/

    data_dir = "/content/my_dataset"
    model_output_dir = "/content/drive/MyDrive/colab/UMUD"

else:

    data_dir = "content/my_dataset"
    model_output_dir = "segmentation_baseline_models"


jst = timezone(timedelta(hours=9))
execution_start_datetime = datetime.now(jst)
execution_start_time = time.perf_counter()

print("=============================================")
print(
    "実行開始時間:",
    execution_start_datetime.strftime("%Y-%m-%d %H:%M:%S JST")
)
print("=============================================")


device_check = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("device:", device_check)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



def load_fasc_binary_mask(mask_dir, file_name):
    """
    fasc mask を
    背景=0
    fascicle=1
    の二値配列に変換する
    """
    mask = Image.open(os.path.join(mask_dir, file_name))
    mask_array = np.array(mask)

    if mask.mode == "L":
        binary_mask = (mask_array == 255).astype(np.uint8)
    else:
        raise ValueError(f"未対応のmodeです: {mask.mode}")

    return binary_mask


def calculate_sha256(file_path):
    """
    ファイルのSHA-256を計算する
    """
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            sha256.update(chunk)
    return sha256.hexdigest()


def resize_with_padding(image, mask, target_size=(768, 512)):
    """
    image と mask の縦横比を維持したまま縮小し、
    target_size まで余白を追加する
    """

    target_width, target_height = target_size
    original_width, original_height = image.size

    scale = min(target_width / original_width, target_height / original_height)
    new_width = int(original_width * scale)
    new_height = int(original_height * scale)


    image = image.resize((new_width, new_height), resample=Image.Resampling.BILINEAR)
    mask = mask.resize((new_width, new_height), resample=Image.Resampling.NEAREST)
    left = (target_width - new_width) // 2
    top = (target_height - new_height) // 2
    image_canvas = Image.new(image.mode, target_size, 0)
    mask_canvas = Image.new("L", target_size, 0)
    image_canvas.paste(image, (left, top))
    mask_canvas.paste(mask, (left, top))

    return image_canvas, mask_canvas


class SegmentationDataset(Dataset):

    def __init__(self, image_dir, mask_dir, mask_loader):

        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.mask_loader = mask_loader

        image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith(
                (".tif", ".tiff", ".png", ".jpg", ".jpeg")
            )
        ])

        #image + mask が完全一致する重複を除去
        unique_image_files = []
        seen_pairs = set()

        for file_name in image_files:

            image_path = os.path.join(image_dir, file_name)
            mask_path = os.path.join(mask_dir, file_name)

            image_hash = calculate_sha256(image_path)
            mask_hash = calculate_sha256(mask_path)

            pair_hash = (image_hash, mask_hash)

            if pair_hash in seen_pairs:
                continue

            seen_pairs.add(pair_hash)
            unique_image_files.append(file_name)

        self.image_files = unique_image_files

        print("=============================================")
        print("""

        From the URL discussion below, there is duplicate data in the official dataset.
        To prevent data leak between train / validation
        Duplicate deletion processed

        下記URLディスカッションより、公式datasetに重複データがあり、
        train / validation間のデータリークを防止する為
        重複削除処理しました。.....

        引用
        https://www.kaggle.com/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/discussion/740356

        """)
        print("Dataset duplicate removal")
        print("before:", len(image_files))
        print("after:", len(self.image_files))
        print("removed:", len(image_files) - len(self.image_files))
        print("=============================================")



    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):

        file_name = self.image_files[index]
        image_path = os.path.join(self.image_dir, file_name)
        image = Image.open(image_path).convert("RGB")
        mask = self.mask_loader(self.mask_dir, file_name)

        #maskとimageをサイズ揃える
        mask = Image.fromarray(mask)
        mask = mask.resize(image.size, resample=Image.Resampling.NEAREST)

        image, mask = resize_with_padding(image, mask, target_size=(768, 512))

        # image → numpy → Tensor
        image = np.array(image)
        image = torch.from_numpy(image).float() / 255.0
        image = image.permute(2, 0, 1)

        # mask → numpy → Tensor
        mask = np.array(mask)
        mask = torch.from_numpy(mask).float()
        mask = mask.unsqueeze(0)

        return image, mask


class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # 出力
        self.output = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):

        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        enc3 = self.enc3(self.pool2(enc2))
        enc4 = self.enc4(self.pool3(enc3))

        bottleneck = self.bottleneck(self.pool4(enc4))

        dec4 = self.up4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.dec4(dec4)

        dec3 = self.up3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.dec3(dec3)

        dec2 = self.up2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.dec2(dec2)

        dec1 = self.up1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.dec1(dec1)

        return self.output(dec1)

class TverskyLoss(nn.Module):

    def __init__(self, alpha=0.3, beta=0.7):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, outputs, masks):
        pred_probs = torch.sigmoid(outputs)

        tp = (pred_probs * masks).sum()
        fp = (pred_probs * (1 - masks)).sum()
        fn = ((1 - pred_probs) * masks).sum()
        tversky = tp / (tp + self.alpha * fp + self.beta * fn)

        return 1 - tversky

# Dataset作成 → 学習 → 保存 → Dice
def run_segmentation(
    image_dir,
    mask_dir,
    mask_loader,
    model_save_name,
    label_name,
    thresholds,
    global_best_dice,
    lr=0.001,
    batch_size=2,
    num_epochs=50,
    pos_weight=None
):

    print()
    print("=============================================")
    print(f"{label_name} segmentation")
    print("=============================================")

    # Dataset

    dataset = SegmentationDataset(image_dir, mask_dir, mask_loader)
    train_size = int(len(dataset) * 0.8)
    val_size = (len(dataset) - train_size)

    generator = (torch.Generator().manual_seed(42))
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


    #設定

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print("device:", device)

    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    #モデル

    model = UNet()
    model = model.to(device)

    if pos_weight is None:
        bce_criterion = nn.BCEWithLogitsLoss()
    else:
        pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=device)
        bce_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
        print("pos_weight:", pos_weight)

    dice_criterion = TverskyLoss(alpha=0.3, beta=0.7)

    optimizer = torch.optim.Adam(
        model.parameters(),                             #U-Netの学習対象パラメータ
        lr=lr                                           #1回の更新幅
    )

    best_val_dice = -1
    best_val_loss = None
    best_train_loss = None
    best_epoch = 0
    best_threshold = None


    #学習

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for images, masks in train_loader:

            images = images.to(device)
            masks = masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)

            loss = bce_criterion(outputs, masks) + dice_criterion(outputs, masks)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_loss = (total_loss / len(train_loader))

        # validation

        model.eval()

        val_total_loss = 0

        intersection_totals = {
            threshold: 0
            for threshold in thresholds
        }

        pred_totals = {
            threshold: 0
            for threshold in thresholds
        }

        true_total = 0


        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)
                outputs = model(images)

                loss = bce_criterion(outputs, masks) + dice_criterion(outputs, masks)

                val_total_loss += loss.item()
                pred_probs = torch.sigmoid(outputs)

                for threshold in thresholds:
                    pred_masks = (pred_probs >= threshold).float()

                    intersection_totals[threshold] += (pred_masks * masks).sum().item()
                    pred_totals[threshold] += pred_masks.sum().item()

                true_total += masks.sum().item()

        val_loss = (val_total_loss / len(val_loader))

        epoch_best_threshold = None
        epoch_best_dice = -1

        for threshold in thresholds:

            dice = (2 * intersection_totals[threshold]) / (pred_totals[threshold] + true_total)

            if dice > epoch_best_dice:
                epoch_best_threshold = threshold
                epoch_best_dice = dice

        val_dice = epoch_best_dice

        print("=============================================")
        print(
            f"{label_name} "
            f"epoch: {epoch + 1}/{num_epochs}"
        )
        print("train loss:", train_loss)
        print("validation loss:", val_loss)
        print("validation Dice:", val_dice)
        print("validation best threshold:", epoch_best_threshold)

        #この条件内でDice最大の結果を保存

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_val_loss = val_loss
            best_train_loss = train_loss
            best_epoch = epoch + 1
            best_threshold = epoch_best_threshold

        if val_dice > global_best_dice:
            global_best_dice = val_dice

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "best_threshold": epoch_best_threshold,
                    "best_validation_dice": global_best_dice,
                    "best_epoch": epoch + 1,
                    "lr": lr,
                    "pos_weight": pos_weight
                },
                model_save_name
            )

            print()
            print("***** GLOBAL BEST UPDATE *****")
            print("model:", model_save_name)
            print("Dice:", global_best_dice)
            print("threshold:", epoch_best_threshold)
            print("epoch:", epoch + 1)
            print("lr:", lr)
            print("pos_weight:", pos_weight)
            print("******************************")

    # 学習結果

    print(f"""
=================================================
{label_name}
lr: {lr}
pos_weight: {pos_weight}
train loss: {best_train_loss}
validation loss: {best_val_loss}
best epoch: {best_epoch}
best validation Dice: {best_val_dice}
best threshold: {best_threshold}
=================================================
""")


    return {
        "label_name": label_name,
        "train_loss": best_train_loss,
        "validation_loss": best_val_loss,
        "best_epoch": best_epoch,
        "best_validation_dice": best_val_dice,
        "best_threshold": best_threshold,
        "dice": best_val_dice,
        "lr": lr,
        "pos_weight": pos_weight,
        "global_best_dice": global_best_dice
    }


#_/_/_/_/_/_/_/_/_/_/_/_/fasc_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/


fasc_images_new_model_v1 = os.path.join(data_dir, "fasc_imgs_v1", "fasc_images_new_model_v1")
fasc_masks_new_model_v1 = os.path.join(data_dir, "fasc_masks_v1", "fasc_masks_new_model_v1")

fasc_thresholds = [0.91, 0.93]

fasc_pos_weight = 10

fasc_results = []
fasc_best_result = None
fasc_global_best_dice = -1

model_save_name = os.path.join(model_output_dir, "fasc_model.pth")

fasc_lrs = [0.0006]

for fasc_lr in fasc_lrs:

    label_name = f"fasc lr={fasc_lr} pos_weight={fasc_pos_weight}"

    fasc_result = run_segmentation(
        image_dir=fasc_images_new_model_v1,
        mask_dir=fasc_masks_new_model_v1,
        mask_loader=load_fasc_binary_mask,
        model_save_name=model_save_name,
        label_name=label_name,
        thresholds=fasc_thresholds,
        global_best_dice=fasc_global_best_dice,
        lr=fasc_lr,
        batch_size=2,
        num_epochs=40,
        pos_weight=fasc_pos_weight
    )

    fasc_global_best_dice = fasc_result["global_best_dice"]
    fasc_results.append(fasc_result)

    if (
        fasc_best_result is None
        or fasc_result["best_validation_dice"] > fasc_best_result["best_validation_dice"]
    ):
        fasc_best_result = fasc_result


#_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/_/

print()
print("=============================================")
print("FASC BEST RESULT")
print("=============================================")
print("lr:", fasc_best_result["lr"])
print("pos_weight:", fasc_best_result["pos_weight"])
print("best epoch:", fasc_best_result["best_epoch"])
print("best threshold:", fasc_best_result["best_threshold"])
print("validation loss:", fasc_best_result["validation_loss"])
print("best validation Dice:", fasc_best_result["best_validation_dice"])
print("model:", os.path.join(model_output_dir, "fasc_model.pth"))
print("=============================================")


#時間確認

execution_end_datetime = datetime.now(jst)
execution_elapsed_seconds = time.perf_counter() - execution_start_time

hours = int(execution_elapsed_seconds // 3600)
minutes = int((execution_elapsed_seconds % 3600) // 60)
seconds = int(execution_elapsed_seconds % 60)

print()
print("=============================================")
print(
    "実行開始時間:",
    execution_start_datetime.strftime("%Y-%m-%d %H:%M:%S JST")
)
print(
    "すべての処理が終了した時間:",
    execution_end_datetime.strftime("%Y-%m-%d %H:%M:%S JST")
)
print(
    "総処理時間:",
    f"{hours:02d}:{minutes:02d}:{seconds:02d}"
)
print("=============================================")


if is_colab:

    print("Google Driveへの書き込みを確定します")
    drive.flush_and_unmount()

    print("Google Driveへの保存が完了しました")
    print("Colabランタイムを切断します")

    runtime.unassign()

Mounted at /content/drive
実行開始時間: 2026-09-26 21:49:02 JST
device: cuda
GPU: NVIDIA A100-SXM4-80GB

fasc lr=0.0006 pos_weight=10 segmentation


        From the URL discussion below, there is duplicate data in the official dataset.
        To prevent data leak between train / validation
        Duplicate deletion processed

        下記URLディスカッションより、公式datasetに重複データがあり、
        train / validation間のデータリークを防止する為
        重複削除処理しました。.....

        引用
        https://www.kaggle.com/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/discussion/740356

        
Dataset duplicate removal
before: 2761
after: 2091
removed: 670
device: cuda
pos_weight: 10
fasc lr=0.0006 pos_weight=10 epoch: 1/40
train loss: 0.9685022884436201
validation loss: 0.8815700187569573
validation Dice: 0.20703925363827047
validation best threshold: 0.91

***** GLOBAL BEST UPDATE *****
model: /content/drive/MyDrive/colab/UMUD/fasc_model.pth
Dice: 0.20703925363827047
threshold: 0.91
epoch: 1
lr: 0.0006
pos_weig